# FringeNet — Colab GPU 학습 드라이버 (VS Code Colab extension)

`src/train_gpu.py`(CUDA, holdout 전용)를 구동하는 노트북. CPU 파이프라인(`src/train.py`)과 디커플되어 있고, 산출물 계약(`runs/<experiment>/<run_name>/{model.pt, train.log, metrics.json}`)은 동일하다.

**사용법**: IDE에서 이 노트북을 열고, 커널 선택에서 [Colab extension](https://marketplace.visualstudio.com/items?itemName=Google.colab)의 GPU 런타임에 연결한 뒤 위에서부터 실행한다. 노트북 파일과 셀 출력은 로컬에 저장된다.

**전제 — 커널의 파일시스템은 Colab VM이다.** extension은 코드 실행만 원격으로 보낼 뿐 로컬 워크스페이스 파일을 동기화하지 않는다. 따라서 `src/`·`configs/`·데이터는 VM 쪽에 준비돼야 하고(셀 1이 clone/pull로 처리), `runs/` 산출물도 VM 디스크에 생기므로 **push로 회수하지 않으면 런타임 종료와 함께 사라진다**.

**반복 루프**:
1. 로컬에서 코드·config 수정 → commit & **push** (VM은 origin에서 코드를 받는다)
2. 셀 1 재실행으로 VM의 clone을 `git pull` 최신화 — 코드가 바뀌었으면 **커널 재시작**으로 import 캐시를 비운 뒤 다시 위에서부터
3. 학습 실행 → 마지막 셀에서 결과 commit & push
4. 로컬에서 `git pull` → `evaluate.py`·리포트로 분석 → 다음 태스크

**사전 준비 (최초 1회)**: 대회 CSV 3종을 본인 Google Drive `FringeNet/data/`에 업로드, push용 GitHub fine-grained PAT(Contents: Read and write) 발급.

In [ ]:
# 셀 1 — 런타임 준비: Colab VM에 저장소 clone(최초) 또는 pull(재실행 시 최신화)
# 커널 파일시스템은 VM이라 로컬 워크스페이스가 보이지 않는다 — 코드는 origin 기준.
# 로컬 커널로 연결했을 때는 clone 없이 저장소 루트로만 이동한다 (CPU 폴백 확인용).
import sys
from pathlib import Path

REPO_URL = "https://github.com/SungHan-Bae/FringeNet.git"

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_root = Path("/content/FringeNet")
    if repo_root.exists():
        # 로컬에서 push한 최신 코드 반영 — 코드가 바뀌었으면 커널 재시작 후 재실행
        !git -C {repo_root} pull --ff-only
    else:
        !git clone {REPO_URL} {repo_root}
    !git -C {repo_root} log --oneline -1
else:
    # 로컬 커널: 이 노트북(notebooks/)의 상위에서 저장소 루트를 찾는다
    repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())

%cd {repo_root}
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo:", repo_root, "| Colab VM 커널:", IN_COLAB)

In [ ]:
# GPU 확인 — Colab 프리셋 torch는 CUDA 빌드라 별도 설치가 필요 없다 (재설치 금지)
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("경고: GPU 미연결 — 런타임 유형을 GPU로 변경할 것 (CPU로도 돌지만 매우 느리다)")

In [ ]:
# 데이터 준비 — data/raw/의 대회 CSV는 git에 없다 (재배포 금지 계약, CLAUDE.md)
# Colab VM: Drive에서 복사. 로컬: 이미 있어야 한다.
import shutil

DRIVE_DATA_DIR = "/content/drive/MyDrive/FringeNet/data"  # 본인 Drive 경로에 맞게 조정

raw_dir = repo_root / "data" / "raw"
needed = ["train.csv", "test.csv", "sample_submission.csv"]
missing = [f for f in needed if not (raw_dir / f).exists()]

if missing and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    raw_dir.mkdir(parents=True, exist_ok=True)
    for f in missing:
        src = Path(DRIVE_DATA_DIR) / f
        assert src.exists(), f"Drive에 {src} 가 없다 — 대회 데이터를 먼저 올려둘 것"
        shutil.copy(src, raw_dir / f)
        print("복사:", f)

missing = [f for f in needed if not (raw_dir / f).exists()]
assert not missing, f"data/raw/ 에 누락: {missing}"
print("데이터 준비 완료 — parquet 캐시는 최초 로드 시 자동 생성(약 29초)")

In [ ]:
# 이번 세션에서 돌릴 실험 목록
# SMOKE=True: subset 2만 행 x 2 epoch으로 파이프라인만 검증 (run_name에 -smoke 접미사,
# 본 실험 산출물을 덮어쓰지 않는다). 본 학습 전에 한 번 켜서 확인할 것.
CONFIGS = [
    "configs/level1_cnn/single-scale.yaml",
    "configs/level1_cnn/single-scale-shuffled.yaml",
]
SMOKE = True

In [ ]:
# 학습 실행 — 진행 로그는 에폭마다 즉시 runs/<experiment>/<run_name>/train.log에도 남는다
from src.train_gpu import run_config

all_metrics = {}
for cfg_path in CONFIGS:
    print(f"\n=== {cfg_path} ===")
    overrides = (
        {"subset": 20_000, "epochs": 2, "run_name": Path(cfg_path).stem + "-smoke"}
        if SMOKE
        else {}
    )
    all_metrics[cfg_path] = run_config(cfg_path, **overrides)

In [ ]:
# holdout MAE 요약 — CPU baseline(고정 기준선)과 나란히
BASELINE = ("mlp_baseline/dropout0.0 (CPU, Task 4)", 4.599)

print(f"{'run':48s}  holdout MAE [nm]")
print(f"{BASELINE[0]:48s}  {BASELINE[1]:.4f}")
for m in all_metrics.values():
    r = m["model"]
    per_layer = r["val_mae_per_layer"]
    layers = "  ".join(f"L{i}={per_layer[f'layer_{i}']:.3f}" for i in range(1, 5))
    name = f"{m['experiment']}/{m['run_name']}"
    print(f"{name:48s}  {r['val_mae']:.4f}  ({layers}, best ep {r['best_epoch']})")

In [ ]:
# 결과 commit & push — Colab VM에서 실행했을 때만 필요 (로컬 커널이면 이미 로컬에 있다)
# PAT는 push URL에만 쓰고 .git/config에 저장하지 않는다. 스모크(-smoke) 산출물은 push 대상 아님.
if IN_COLAB and not SMOKE:
    from getpass import getpass

    author = !git log -1 --format=%an
    email = !git log -1 --format=%ae
    !git config user.name "{author[0]}"
    !git config user.email "{email[0]}"
    !git add runs/
    !git status --short
    !git commit -m "exp: level1_cnn GPU 학습 결과 (Colab)"
    token = getpass("GitHub PAT (Contents: Read and write): ")
    !git push https://{token}@github.com/SungHan-Bae/FringeNet.git HEAD:main
    del token
else:
    print("push 생략 —", "스모크 실행" if SMOKE else "로컬 커널 (산출물이 이미 로컬 저장소에 있음)")

## 다음 단계 (로컬에서)

```bash
git pull
python -m src.evaluate --run runs/level1_cnn/single-scale   # 상세 분석
```

분석·리포트 취합(`reports/level1_cnn.md`)과 주차 노트(`docs/week_1.md`) 갱신은 로컬 Claude Code 세션에서 진행.